# Data merge and feature engineering


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [2]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw = pd.read_csv("../data/isw_processed.csv")
df_isw_matrix = pd.read_csv("../data/isw_processed_svd.csv")
df_tg = pd.read_csv("../data/telegram_processed.csv")

In [3]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [4]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,region_key,region_id,alarm_active,alarm_minutes_in_hour
33815,50.4506,30.5243,Kyiv,Europe/Kiev,2022-04-23,10.8,5.8,72.7,0.500,100.0,...,True,False,False,True,False,False,Київ,26,0,0.000000
596573,48.6264,22.2851,Uzhgorod,Europe/Uzhgorod,2024-12-25,0.7,-3.5,74.1,0.090,100.0,...,False,True,True,False,False,False,Закарпатська,7,0,0.000000
36555,48.0020,37.8145,Donetsk,Europe/Kiev,2022-04-28,14.7,5.6,56.2,0.000,0.0,...,False,False,False,True,False,False,Донецька,5,0,0.000000
281680,49.5527,25.5889,Ternopil,Europe/Kiev,2023-06-28,15.7,10.7,73.6,1.800,100.0,...,True,False,False,True,False,False,Тернопільська,19,0,0.000000
259758,47.8289,35.1626,Zaporozhye,Europe/Zaporozhye,2023-05-21,18.0,10.9,64.3,0.300,100.0,...,True,False,False,True,False,False,Запорізька,8,0,0.000000
237166,51.4937,31.2944,Chernihiv,Europe/Kiev,2023-04-11,10.9,1.9,58.8,0.000,0.0,...,False,False,False,True,False,False,Чернігівська,25,0,0.000000
26591,50.4506,30.5243,Kyiv,Europe/Kiev,2022-04-11,6.7,-0.2,62.7,3.000,100.0,...,True,False,False,True,False,False,Київ,26,1,26.016667
14625,48.5085,32.2656,Kropyvnytskyi,Europe/Kiev,2022-03-21,3.9,-3.0,62.6,0.000,0.0,...,False,False,True,False,False,False,Кіровоградська,11,0,0.000000
22487,50.4506,30.5243,Kyiv,Europe/Kiev,2022-04-04,1.6,-5.6,61.8,0.000,0.0,...,False,False,False,True,False,False,Київ,26,0,0.000000
502857,48.5085,32.2656,Kropyvnytskyi,Europe/Kiev,2024-07-16,30.5,17.6,49.3,0.000,0.0,...,False,False,True,False,False,False,Кіровоградська,11,1,45.350000


In [5]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,region_key,region_id,alarm_active,alarm_minutes_in_hour
0,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0
24,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0
48,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0
72,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0
96,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0


In [6]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 634752 entries, 0 to 634751
Data columns (total 56 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   city_latitude                  634752 non-null  float64       
 1   city_longitude                 634752 non-null  float64       
 2   city_name                      634752 non-null  str           
 3   city_timezone                  634752 non-null  str           
 4   day_datetime                   634752 non-null  str           
 5   day_temp                       634752 non-null  float64       
 6   day_dew                        634752 non-null  float64       
 7   day_humidity                   634752 non-null  float64       
 8   day_precip                     634752 non-null  float64       
 9   day_precipprob                 634752 non-null  float64       
 10  day_precipcover                634752 non-null  float64       
 11  day_snow        

In [7]:
df_final['alarms_in_last_24h'] = df_final.groupby('region_id')['alarm_active'].rolling(window=24, min_periods=1).sum().reset_index(level=0, drop=True)
df_final['alarm_in_last_hour'] = df_final.groupby('region_id')['alarm_active'].shift(1)
df_final['total_active_alarms'] = df_final.groupby('datetime_hour')['alarm_active'].transform('sum')
df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

df_final.sample(10)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,hour_conditions_simple_Snow,region_key,region_id,alarm_active,alarm_minutes_in_hour,alarms_in_last_24h,alarm_in_last_hour,total_active_alarms,is_weekend,is_night
49221,48.2924,25.9355,Chernivtsi,Europe/Kiev,2022-05-20,18.8,2.2,36.8,0.000,0.0,...,False,Чернівецька,24,0,0.00,0.0,0.0,4,0,0
325801,50.7469,25.3263,Lutsk,Europe/Kiev,2023-09-12,19.5,13.1,69.7,0.000,0.0,...,False,Волинська,3,0,0.00,0.0,0.0,3,0,0
459736,49.5527,25.5889,Ternopil,Europe/Kiev,2024-05-02,17.0,4.9,47.4,0.000,0.0,...,False,Тернопільська,19,0,0.00,0.0,0.0,1,0,1
261807,50.9080,34.7972,Sumy,Europe/Kiev,2023-05-24,17.7,13.9,79.5,0.705,100.0,...,False,Сумська,18,0,0.00,7.0,0.0,11,0,0
204156,46.4725,30.7371,Odesa,Europe/Kiev,2023-02-13,1.2,-5.3,63.0,0.000,0.0,...,False,Одеська,15,0,0.00,0.0,0.0,0,0,0
264663,50.9080,34.7972,Sumy,Europe/Kiev,2023-05-29,15.5,5.4,53.9,0.000,0.0,...,False,Сумська,18,1,34.70,9.0,1.0,21,0,0
278076,46.4725,30.7371,Odesa,Europe/Kiev,2023-06-21,23.1,15.2,61.8,1.700,100.0,...,False,Одеська,15,0,0.00,0.0,0.0,0,0,0
458765,48.6264,22.2851,Uzhgorod,Europe/Uzhgorod,2024-04-30,16.9,6.3,53.3,0.000,0.0,...,False,Закарпатська,7,0,0.00,0.0,0.0,15,0,0
155848,49.5527,25.5889,Ternopil,Europe/Kiev,2022-11-21,-0.5,-1.2,94.9,1.300,100.0,...,False,Тернопільська,19,0,0.00,0.0,0.0,0,0,0
514730,48.4753,35.0160,Dnipro,Europe/Kiev,2024-08-05,21.8,14.3,65.6,0.400,100.0,...,False,Дніпропетровська,4,1,24.35,9.0,1.0,15,0,0


In [8]:
"""neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(index='datetime_hour', columns='region_id', values='alarm_active', fill_value=0)

def get_neighbour_sum_safe(row):
    region_id = row['region_id']
    datetime = row['datetime_hour']
    
    if region_id in neighbouring_regions:
        neighbours = neighbouring_regions[region_id]
        
        if datetime in alarms_matrix.index:
            valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]

            if valid_neighbours:
                return alarms_matrix.loc[datetime, valid_neighbours].sum()
    return 0

df_final['neighbour_alarms'] = df_final.apply(get_neighbour_sum_safe, axis=1)

df_final.sample(15)"""

"neighbouring_regions = {\n    1: [21],\n    2: [6, 10, 11, 15, 22, 23, 24],\n    3: [13, 17],\n    4: [5, 8, 11, 14, 16, 20, 21],\n    5: [4, 8, 12, 20],\n    6: [2, 10, 17, 22],\n    7: [9, 13],\n    8: [4, 5, 21],\n    9: [7, 13, 19, 24],\n    10: [2, 6, 16, 23, 25],\n    11: [2, 4, 14, 15, 16, 23],\n    12: [5, 20],\n    13: [3, 7, 9, 17, 19],\n    14: [4, 11, 15, 21],\n    15: [2, 11, 14],\n    16: [4, 10, 11, 18, 20, 23, 25],\n    17: [3, 6, 13, 19, 22],\n    18: [16, 20, 25],\n    19: [9, 13, 17, 22, 24],\n    20: [4, 5, 12, 16, 18],\n    21: [1, 4, 8, 14],\n    22: [2, 6, 17, 19, 24],\n    23: [2, 10, 11, 16],\n    24: [2, 9, 19, 22],\n    25: [10, 16, 18], \n    26: [10]\n}\n\nalarms_matrix = df_final.pivot_table(index='datetime_hour', columns='region_id', values='alarm_active', fill_value=0)\n\ndef get_neighbour_sum_safe(row):\n    region_id = row['region_id']\n    datetime = row['datetime_hour']\n\n    if region_id in neighbouring_regions:\n        neighbours = neighbouring_

In [9]:
def calculate_hours_since_last(series):
    groups = series.cumsum()
    return series.groupby(groups).cumcount()

df_final['hours_since_last_alarm'] = df_final.groupby('region_id')['alarm_active'].transform(calculate_hours_since_last)

In [10]:
"""df_final['neighbour_alarms'] = df_final['neighbour_alarms'].astype(int)
df_final['alarms_in_last_24h'] = df_final['alarms_in_last_24h'].astype(int)
df_final['alarm_in_last_hour'] = df_final['alarm_in_last_hour'].fillna(0).astype(int)"""

"df_final['neighbour_alarms'] = df_final['neighbour_alarms'].astype(int)\ndf_final['alarms_in_last_24h'] = df_final['alarms_in_last_24h'].astype(int)\ndf_final['alarm_in_last_hour'] = df_final['alarm_in_last_hour'].fillna(0).astype(int)"

In [11]:
df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
df_final["alarm_lag_6"] = df_final.groupby("region_id")["alarm_active"].shift(6)
df_final["alarm_lag_12"] = df_final.groupby("region_id")["alarm_active"].shift(12)

In [12]:
df_final.sample(20)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,alarms_in_last_24h,alarm_in_last_hour,total_active_alarms,is_weekend,is_night,hours_since_last_alarm,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12
39122,48.47530,35.01600,Dnipro,Europe/Kiev,2022-05-02,12.9,2.8,52.2,7.790,100.0,...,6.0,0.0,0,0,1,7,0.0,0.0,0.0,0.0
509873,50.00420,36.23580,Kharkiv,Europe/Kiev,2024-07-28,25.1,12.6,47.5,0.000,0.0,...,24.0,1.0,7,1,1,0,1.0,1.0,1.0,1.0
39105,48.50850,32.26560,Kropyvnytskyi,Europe/Kiev,2022-05-02,11.8,2.7,57.1,0.000,0.0,...,1.0,0.0,0,0,0,20,0.0,0.0,0.0,0.0
91365,48.29240,25.93550,Chernivtsi,Europe/Kiev,2022-08-01,18.0,16.3,89.8,13.108,100.0,...,2.0,0.0,0,0,0,16,0.0,0.0,0.0,0.0
573414,47.82890,35.16260,Zaporozhye,Europe/Zaporozhye,2024-11-15,3.3,-0.2,77.7,0.100,100.0,...,6.0,1.0,3,0,0,1,1.0,0.0,0.0,0.0
420036,46.47250,30.73710,Odesa,Europe/Kiev,2024-02-23,7.4,5.0,85.1,0.000,0.0,...,6.0,0.0,0,0,1,6,0.0,0.0,1.0,0.0
133456,49.55270,25.58890,Ternopil,Europe/Kiev,2022-10-13,9.2,4.9,75.7,0.000,0.0,...,4.0,1.0,1,0,0,1,1.0,0.0,0.0,0.0
293960,50.45060,30.52430,Kyiv,Europe/Kiev,2023-07-19,20.2,10.4,56.9,0.000,0.0,...,7.0,0.0,3,0,0,2,0.0,1.0,1.0,0.0
203324,49.44070,32.06370,Cherkasy,Europe/Kiev,2023-02-11,0.0,-4.1,74.0,0.200,100.0,...,1.0,0.0,5,1,1,23,0.0,0.0,0.0,0.0
127163,46.97336,31.98522,Mykolaiv,Europe/Kiev,2022-10-02,17.5,14.1,80.6,15.900,100.0,...,7.0,0.0,3,1,0,0,0.0,0.0,0.0,0.0


In [13]:
df_isw_matrix.sample(10)

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
674,2024-01-04,0.733938,-0.027244,-0.221954,0.086682,0.176155,0.119486,-0.043461,-0.086818,0.070242,...,-0.012019,0.002467,-0.015473,-0.001946,0.044419,-0.025504,-0.003577,-0.022111,0.040944,0.010031
696,2024-01-26,0.721717,-0.045088,-0.301852,0.149813,0.175023,0.134555,-0.049920,-0.078168,0.049811,...,0.009189,-0.014163,0.006726,-0.031137,-0.008190,0.023955,0.026988,-0.005236,0.004314,0.008057
470,2023-06-11,0.774789,-0.162893,-0.028728,-0.092384,-0.112150,0.102778,0.029590,-0.002814,-0.115976,...,-0.008336,0.036908,-0.007083,0.000687,-0.008124,-0.004046,-0.001901,-0.026735,-0.006051,-0.021118
639,2023-11-28,0.776683,-0.098416,-0.154486,-0.133516,0.041368,-0.178677,0.110448,-0.118916,0.140886,...,-0.016728,-0.024516,0.006396,0.032991,-0.024617,-0.003381,-0.017932,0.012893,0.014739,0.006208
719,2024-02-18,0.825742,-0.067896,-0.099098,0.062079,0.053547,-0.012352,-0.027162,0.013981,0.007317,...,0.037139,0.009011,-0.008229,-0.000395,-0.003304,0.002229,-0.014349,-0.028010,-0.005468,-0.000807
148,2022-07-21,0.699001,-0.182578,0.263780,-0.016603,-0.108189,0.153123,0.085143,-0.114858,-0.025426,...,0.030896,0.008585,-0.032267,0.023466,0.035215,-0.011963,-0.021054,0.009809,-0.016191,0.013751
288,2022-12-09,0.801194,-0.192327,-0.092605,-0.075411,0.123440,-0.050112,0.179692,0.071591,-0.073356,...,-0.014852,0.005112,-0.000156,0.004379,0.019923,-0.008044,0.025054,0.010016,0.036576,-0.008917
978,2024-11-04,0.741363,0.062600,-0.074753,-0.069774,0.008951,-0.225485,0.093867,-0.184096,0.233085,...,0.036599,0.018105,-0.006530,0.014879,-0.032924,0.015505,-0.004663,-0.009581,-0.021580,-0.006038
182,2022-08-24,0.724540,-0.225303,0.194534,-0.029100,-0.071329,0.096870,0.014818,0.040877,0.122164,...,0.015849,0.029439,-0.008829,0.015789,-0.023049,-0.054872,-0.023790,0.000789,0.013092,-0.020356
298,2022-12-19,0.754575,-0.230595,-0.042941,-0.106010,0.173241,-0.003807,0.260485,0.115314,-0.107389,...,0.007249,0.042209,0.024083,0.008687,0.032186,-0.009856,-0.014857,0.009568,0.002033,-0.017796


In [14]:
df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [15]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime']).dt.date
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime']).dt.date
df_isw_matrix = df_isw_matrix.fillna(0, inplace=True)
df_isw_matrix.isna().sum()

day_datetime     0
isw_topic_0      0
isw_topic_1      0
isw_topic_2      0
isw_topic_3      0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 151, dtype: int64

In [16]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [17]:
df_final.sample(10)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
527963,49.41680,26.97430,Khmelnytskyi,Europe/Kiev,2024-12-30,0.1,-1.4,89.8,0.00,0.0,...,0.015635,0.001037,0.010634,0.014117,-0.025537,0.006101,-0.033841,0.024780,0.002579,0.014878
433359,49.55270,25.58890,Ternopil,Europe/Kiev,2023-04-07,3.8,1.7,86.7,0.40,100.0,...,0.019287,0.006004,0.024249,0.023730,0.003923,0.007818,-0.020718,-0.010000,-0.014954,-0.025220
79070,48.47530,35.01600,Dnipro,Europe/Kiev,2025-02-15,-2.5,-4.2,87.8,9.00,100.0,...,0.001261,-0.007862,-0.007320,-0.013219,-0.010327,-0.009311,-0.026183,0.021919,0.005124,0.001581
416002,50.90800,34.79720,Sumy,Europe/Kiev,2024-04-21,14.2,7.8,66.5,6.41,100.0,...,-0.036378,-0.012886,0.040736,-0.008357,0.007613,-0.024142,-0.046724,-0.005802,0.032071,-0.002289
186669,48.92260,24.71470,Ivano-Frankivsk,Europe/Kiev,2022-04-20,5.5,-3.2,57.0,0.10,100.0,...,0.028549,-0.018930,0.007139,0.006329,-0.007923,-0.021398,0.005896,0.016074,-0.024424,0.033287
402701,50.90800,34.79720,Sumy,Europe/Kiev,2022-10-15,4.6,0.0,74.7,0.00,0.0,...,-0.001899,-0.029223,0.032751,-0.026982,-0.007735,-0.035608,0.019567,-0.034117,-0.030260,-0.004586
124732,50.25360,28.66540,Zhytomyr,Europe/Kiev,2024-04-18,6.5,4.8,88.8,10.00,100.0,...,-0.022626,-0.002019,0.016278,0.015112,-0.001772,-0.009027,0.032066,0.028168,-0.063069,-0.019673
376865,50.61927,26.25131,Rivne,Europe/Kiev,2022-11-10,7.7,6.3,91.0,1.00,100.0,...,0.038719,-0.012876,-0.013759,-0.020383,-0.021288,0.000479,0.007099,-0.025219,0.051237,-0.017991
24612,49.23360,28.44860,Vinnytsia,Europe/Kiev,2024-12-14,-7.5,-10.3,81.7,0.10,100.0,...,0.008479,0.001538,-0.019994,-0.010032,-0.004889,-0.001505,0.011015,0.012946,-0.022036,0.010253
104058,48.00200,37.81450,Donetsk,Europe/Kiev,2024-12-15,-0.4,-1.9,89.4,3.40,100.0,...,0.022313,0.022430,0.011422,-0.018059,0.005999,-0.002632,-0.021027,-0.006368,-0.012289,0.003041


In [18]:
df_final.isna().sum()

city_latitude        0
city_longitude       0
city_name            0
city_timezone        0
day_datetime         0
                  ... 
isw_topic_145     6336
isw_topic_146     6336
isw_topic_147     6336
isw_topic_148     6336
isw_topic_149     6336
Length: 216, dtype: int64

In [19]:
df_final.fillna(0, inplace=True)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635323,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
635324,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
635325,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
635326,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664


In [20]:
df_final.isna().sum()

city_latitude     0
city_longitude    0
city_name         0
city_timezone     0
day_datetime      0
                 ..
isw_topic_145     0
isw_topic_146     0
isw_topic_147     0
isw_topic_148     0
isw_topic_149     0
Length: 216, dtype: int64

### Feature engineering for isw topics 

In [21]:
topic_cols = [c for c in df_final.columns if "isw_topic_" in c]
df_isw_abs = df_final[topic_cols].abs()

df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
df_final['isw_topic_max'] = df_isw_abs.max(axis=1)
df_final["isw_topic_mean"] = df_isw_abs.mean(axis=1)

df_final['isw_velocity_24h'] = df_final['isw_total_intensity'].diff(24)

df_final["isw_intensity_ema"] = (df_final.groupby("region_id")["isw_total_intensity"].transform(lambda x: x.ewm(span=24).mean()))

df_final["isw_topic_entropy"] = -(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) * np.log(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

"""df_final.drop(columns=topic_cols, inplace=True)
df_final = df_final.copy()"""
df_final.fillna(0, inplace=True)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_18476\1245980488.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_18476\1245980488.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_18476\1245980488.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy
0,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,-0.000000
1,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,-0.000000
2,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,-0.000000
3,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,-0.000000
4,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,-0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635323,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.008404,-0.019102,0.004664,4.222493,0.072108,0.78631,0.02815,-0.455369,4.307044,4.025557
635324,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.008404,-0.019102,0.004664,4.222493,0.072108,0.78631,0.02815,-0.455369,4.300280,4.025557
635325,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.008404,-0.019102,0.004664,4.222493,0.072108,0.78631,0.02815,-0.455369,4.294057,4.025557
635326,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.008404,-0.019102,0.004664,4.222493,0.072108,0.78631,0.02815,-0.455369,4.288332,4.025557


### TELEGRAM MERGE

In [22]:
df_tg_matrix = pd.read_csv("../data/telegram_processed_svd.csv")

In [25]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000990,-0.011395,0.032670,-0.050983,0.061080,...,0.017141,-0.013944,-0.027516,0.034612,-0.032061,-0.051141,-0.004707,-0.003985,-0.052194,-0.026334
1,2026-03-06 12:57:29,DeepStateUA,0.022770,-0.002163,0.041469,0.001527,-0.033584,0.077087,-0.078417,0.091534,...,-0.019097,0.025730,0.000530,0.020332,-0.028041,0.001581,0.006948,-0.064965,0.022597,-0.017810
2,2026-03-06 08:32:47,DeepStateUA,0.009647,-0.001083,0.029820,0.004092,-0.021362,0.053647,-0.076998,0.085930,...,0.025378,-0.003550,-0.043415,-0.007200,-0.009172,0.006124,-0.004550,0.020576,0.010544,0.012825
3,2026-03-05 16:46:58,DeepStateUA,0.006857,0.000279,0.019019,0.000875,-0.011544,0.032104,-0.049929,0.059696,...,0.008728,-0.007502,-0.021433,0.028223,-0.025323,-0.045761,-0.001697,0.002329,-0.042282,-0.017344
4,2026-03-05 15:46:39,DeepStateUA,0.048850,-0.006687,0.076040,-0.003924,-0.027197,0.089562,-0.088811,0.109836,...,0.017978,0.027453,-0.019744,0.008659,0.023139,-0.009485,-0.030263,0.013689,-0.022727,0.006843


In [28]:
df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")

C:\Users\Uw11\AppData\Local\Temp\ipykernel_18476\2851151304.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")


In [29]:
topic_cols = [c for c in df_tg_matrix.columns if "tg_topic_" in c]
tg_hourly = (df_tg_matrix.groupby("datetime_hour")[topic_cols].mean().reset_index())

C:\Users\Uw11\AppData\Local\Temp\ipykernel_18476\873189955.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tg_hourly = (df_tg_matrix.groupby("datetime_hour")[topic_cols].mean().reset_index())


In [30]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,...,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249,datetime_hour
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000990,-0.011395,0.032670,-0.050983,0.061080,...,-0.013944,-0.027516,0.034612,-0.032061,-0.051141,-0.004707,-0.003985,-0.052194,-0.026334,2026-03-06 13:00:00
1,2026-03-06 12:57:29,DeepStateUA,0.022770,-0.002163,0.041469,0.001527,-0.033584,0.077087,-0.078417,0.091534,...,0.025730,0.000530,0.020332,-0.028041,0.001581,0.006948,-0.064965,0.022597,-0.017810,2026-03-06 12:00:00
2,2026-03-06 08:32:47,DeepStateUA,0.009647,-0.001083,0.029820,0.004092,-0.021362,0.053647,-0.076998,0.085930,...,-0.003550,-0.043415,-0.007200,-0.009172,0.006124,-0.004550,0.020576,0.010544,0.012825,2026-03-06 08:00:00
3,2026-03-05 16:46:58,DeepStateUA,0.006857,0.000279,0.019019,0.000875,-0.011544,0.032104,-0.049929,0.059696,...,-0.007502,-0.021433,0.028223,-0.025323,-0.045761,-0.001697,0.002329,-0.042282,-0.017344,2026-03-05 16:00:00
4,2026-03-05 15:46:39,DeepStateUA,0.048850,-0.006687,0.076040,-0.003924,-0.027197,0.089562,-0.088811,0.109836,...,0.027453,-0.019744,0.008659,0.023139,-0.009485,-0.030263,0.013689,-0.022727,0.006843,2026-03-05 15:00:00


In [31]:
tg_hourly["datetime_hour"] = tg_hourly["datetime_hour"] + pd.Timedelta(hours=1)

In [32]:
df_final = df_final.merge(tg_hourly, on="datetime_hour", how="left")

In [33]:
df_final.head()

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
df_final.shape

(635328, 473)

In [2]:
df_final.sample(15)

NameError: name 'df_final' is not defined